# Étape 1 — Extraction & validation de `menu_index.json`

**Objectif** : charger le fichier `menu_index.json` généré par le pipeline existant
(`menu.xml → xml_parser → path_builder → generate_json`), vérifier sa qualité,
et produire un `DataFrame` propre qui servira de **source de vérité unique**
pour la construction du dataset (Étape 2).

**Pourquoi cette étape est nécessaire** :
- Si `menu_index.json` contient des doublons, des chemins vides ou des IDs
  incohérents, ces défauts se propageront dans tout le dataset d'entraînement
  et fausseront le fine-tuning.
- On ne modifie **jamais** les données métier ici : on les *audite* seulement.
  `menu_index.json` reste la source de vérité, on ne fait qu'en dériver une
  version "nettoyée / dédupliquée" pour l'entraînement.

**Ce que ce notebook ne fait PAS** : il ne touche à aucun fichier de
l'application (`chatbot.py`, `rag_chatbot.py`, `semantic_search.py`, ChromaDB
actuel). Tout se passe ici, dans Colab, en local à ce notebook.

## 1.1 — Installation des dépendances

In [10]:
!pip install -q pandas
import pandas as pd
import json
import re
from collections import Counter, defaultdict

pd.set_option("display.max_colwidth", 120)
print("Dépendances chargées.")

Dépendances chargées.


## 1.2 — Charger `menu_index.json`

Deux options :
- **Option A (recommandée)** : uploader le fichier directement dans Colab
  via le bouton "Fichiers" (icône dossier à gauche) puis glisser
  `menu_index.json`, OU utiliser la cellule d'upload ci-dessous.
- **Option B** : monter votre Google Drive si le fichier y est stocké.

La cellule ci-dessous gère les deux cas automatiquement.

In [11]:
import os

MENU_INDEX_PATH = "menu_index.json"

if not os.path.exists(MENU_INDEX_PATH):
    try:
        from google.colab import files
        print("Fichier introuvable localement. Veuillez uploader menu_index.json :")
        uploaded = files.upload()
        # Récupère le premier fichier .json uploadé
        json_files = [f for f in uploaded.keys() if f.endswith(".json")]
        if json_files:
            MENU_INDEX_PATH = json_files[0]
    except ImportError:
        # Environnement non-Colab (ex: test local) -> on suppose le fichier déjà présent
        pass

assert os.path.exists(MENU_INDEX_PATH), (
    f"Fichier {MENU_INDEX_PATH} introuvable. "
    "Uploadez-le ou montez votre Drive avant de continuer."
)
print(f"Fichier trouvé : {MENU_INDEX_PATH}")

Fichier trouvé : menu_index.json


In [12]:
with open(MENU_INDEX_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Type de l'objet racine : {type(raw_data)}")
print(f"Nombre d'entrées brutes : {len(raw_data)}")
print("\nExemple de la 1ère entrée :")
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))

Type de l'objet racine : <class 'list'>
Nombre d'entrées brutes : 1383

Exemple de la 1ère entrée :
{
  "id": "NEM_OUT_004T_1",
  "label": "Test",
  "path_labels": [
    "Parametrage",
    "Test"
  ],
  "path_ids": [
    "PARAMETRAGE",
    "NEM_OUT_004T_1"
  ],
  "path_str": "Parametrage > Test",
  "command_type": null,
  "search_text": "Parametrage > Test"
}


## 1.3 — Vérifier la structure attendue

Chaque entrée doit contenir au minimum les clés suivantes (d'après le schéma
observé dans le projet) :

- `id` (identifiant unique du menu terminal)
- `label` (libellé du dernier niveau)
- `path_labels` (liste des libellés du chemin complet)
- `path_ids` (liste des ids du chemin complet)
- `path_str` (chemin complet lisible, ex: `"Prestations > Referentiel > Actes > Gestion des Actes RO"`)
- `command_type` (peut être `null`)
- `search_text` (texte utilisé pour l'indexation actuelle)

On va vérifier que ce schéma est respecté partout, et signaler toute entrée
non conforme.

In [13]:
EXPECTED_KEYS = {"id", "label", "path_labels", "path_ids", "path_str", "command_type", "search_text"}

schema_issues = []
for i, entry in enumerate(raw_data):
    missing = EXPECTED_KEYS - set(entry.keys())
    extra = set(entry.keys()) - EXPECTED_KEYS
    if missing:
        schema_issues.append((i, entry.get("id", "?"), "missing_keys", missing))
    if extra:
        schema_issues.append((i, entry.get("id", "?"), "extra_keys", extra))

print(f"Nombre d'entrées avec un problème de schéma : {len(schema_issues)}")
for issue in schema_issues[:10]:
    print(" -", issue)

Nombre d'entrées avec un problème de schéma : 0


## 1.4 — Construire le DataFrame et lancer les contrôles qualité

Contrôles effectués :
1. **Chemins vides ou trop courts** (`path_str` vide, ou un seul niveau —
   suspicieux pour un menu métier hiérarchique).
2. **IDs manquants ou dupliqués** (un `id` doit identifier un unique écran).
3. **Chemins dupliqués** (même `path_str` pour deux `id` différents — à
   distinguer d'un `id` dupliqué : cela peut être légitime, ex. deux menus
   pointant vers le même écran physique dans des contextes différents, mais
   on veut le savoir).
4. **Incohérences internes** : `len(path_labels) != len(path_ids)`, ou le
   dernier élément de `path_labels` différent de `label`.
5. **Chemins anormalement longs / courts** (profondeur min/max) pour repérer
   des cas limites.

In [14]:
df = pd.DataFrame(raw_data)

n_total = len(df)

# 1. Chemins vides ou trop courts
df["depth"] = df["path_labels"].apply(lambda x: len(x) if isinstance(x, list) else 0)
empty_paths = df[df["path_str"].fillna("").str.strip() == ""]
shallow_paths = df[df["depth"] <= 1]

# 2. IDs manquants ou dupliqués
missing_ids = df[df["id"].isna() | (df["id"].astype(str).str.strip() == "")]
id_counts = df["id"].value_counts()
duplicated_ids = id_counts[id_counts > 1]

# 3. Chemins dupliqués (path_str identique)
path_counts = df["path_str"].value_counts()
duplicated_paths = path_counts[path_counts > 1]

# 4. Incohérences internes
def check_consistency(row):
    pl, pi, lbl = row["path_labels"], row["path_ids"], row["label"]
    issues = []
    if isinstance(pl, list) and isinstance(pi, list) and len(pl) != len(pi):
        issues.append("len(path_labels) != len(path_ids)")
    if isinstance(pl, list) and len(pl) > 0 and pl[-1] != lbl:
        issues.append("dernier label != label")
    return issues

df["consistency_issues"] = df.apply(check_consistency, axis=1)
inconsistent = df[df["consistency_issues"].apply(len) > 0]

print("=" * 60)
print("RAPPORT DE QUALITÉ — menu_index.json")
print("=" * 60)
print(f"Total d'entrées                         : {n_total}")
print(f"Chemins vides (path_str)                : {len(empty_paths)}")
print(f"Chemins trop courts (depth <= 1)         : {len(shallow_paths)}")
print(f"IDs manquants                           : {len(missing_ids)}")
print(f"IDs dupliqués (valeurs distinctes)       : {len(duplicated_ids)}")
print(f"  -> occurrences totales concernées      : {duplicated_ids.sum() if len(duplicated_ids) else 0}")
print(f"Chemins (path_str) dupliqués (distincts) : {len(duplicated_paths)}")
print(f"  -> occurrences totales concernées      : {duplicated_paths.sum() if len(duplicated_paths) else 0}")
print(f"Entrées incohérentes (labels/ids)        : {len(inconsistent)}")
print(f"Profondeur min / max                    : {df['depth'].min()} / {df['depth'].max()}")
print("=" * 60)

RAPPORT DE QUALITÉ — menu_index.json
Total d'entrées                         : 1383
Chemins vides (path_str)                : 0
Chemins trop courts (depth <= 1)         : 1
IDs manquants                           : 0
IDs dupliqués (valeurs distinctes)       : 38
  -> occurrences totales concernées      : 78
Chemins (path_str) dupliqués (distincts) : 1
  -> occurrences totales concernées      : 2
Entrées incohérentes (labels/ids)        : 0
Profondeur min / max                    : 1 / 4


In [15]:
print("Distribution des profondeurs de chemin :")
print(df["depth"].value_counts().sort_index())

Distribution des profondeurs de chemin :
depth
1      1
2     96
3    462
4    824
Name: count, dtype: int64


In [16]:
if len(duplicated_ids) > 0:
    print("Exemples d'IDs dupliqués :")
    display(df[df["id"].isin(duplicated_ids.index)].sort_values("id")[["id", "path_str"]].head(20))
else:
    print("Aucun ID dupliqué. ✅")

Exemples d'IDs dupliqués :


,id,path_str
1007,COT_APP_005T_1,Cotisations > Editions > Demo Edition Appel Cotisation
992,COT_APP_005T_1,Cotisations > Appel > Lanceur de l'Edition de l'Appel de Cotisation
258,COT_ECH_006T_1,Produits > Referentiel > General > Gestion des Decalages d'echeancier
984,COT_ECH_006T_1,Cotisations > Appel > Gestion des Decalages des Echeanciers
1049,COT_RGT_027T_1,Cotisations > Reglement > Prelevement > Gestion des Prelevements sur Salaire
873,COT_RGT_027T_1,Population > Collectivite > Gestion des Prelevements sur Salaire
1054,PAI_EPA_006T_1,Cotisations > Reglement > Trop Percu > Gestion des Planches de Cheques
239,PAI_EPA_006T_1,Traitements Dossiers > Paiements > Gestion des Planches de Cheques
240,PAI_EPA_007T_1,Traitements Dossiers > Paiements > Renumerotation des cheques
1055,PAI_EPA_007T_1,Cotisations > Reglement > Trop Percu > Renumerotation des Cheques


In [17]:
if len(duplicated_paths) > 0:
    print("Exemples de chemins (path_str) dupliqués :")
    display(df[df["path_str"].isin(duplicated_paths.index)].sort_values("path_str")[["id", "path_str"]].head(20))
else:
    print("Aucun chemin dupliqué. ✅")

Exemples de chemins (path_str) dupliqués :


,id,path_str
1293,PRL_MAN_054T_1,Dossiers Medicaux Administratifs > Gestion des DMA > Validation Manuelle des Prises en Charges
1294,PRL_MAN_054T_3,Dossiers Medicaux Administratifs > Gestion des DMA > Validation Manuelle des Prises en Charges


In [18]:
if len(inconsistent) > 0:
    print("Exemples d'entrées incohérentes :")
    display(inconsistent[["id", "label", "path_str", "consistency_issues"]].head(20))
else:
    print("Aucune incohérence label/path détectée. ✅")

Aucune incohérence label/path détectée. ✅


## 1.5 — Construire le DataFrame "propre" (`df_clean`)

Règles de nettoyage appliquées (conservatrices — on ne supprime que ce qui est
clairement inexploitable pour l'entraînement) :

1. On retire les entrées avec `path_str` vide ou `id` manquant.
2. On **déduplique par `path_str`** en gardant la première occurrence
   (garder le premier `id` rencontré) : un chemin identique ne doit pas
   générer deux exemples différents avec la même cible textuelle.
3. On conserve `path_str` (le chemin complet lisible) comme **cible unique**
   de l'entraînement (`positive`) — c'est ce texte que le modèle doit
   apprendre à rapprocher des questions utilisateur.
4. On garde `id`, `label`, `depth` pour analyse ultérieure (couverture,
   stratification du split, hard negatives par proximité de parent, etc.)

In [19]:
df_clean = df.copy()

# 1. Retirer chemins vides / ids manquants
df_clean = df_clean[df_clean["path_str"].fillna("").str.strip() != ""]
df_clean = df_clean[df_clean["id"].fillna("").astype(str).str.strip() != ""]

# 2. Dédupliquer par path_str (garde la 1ère occurrence)
n_before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["path_str"], keep="first").reset_index(drop=True)
n_after_dedup = len(df_clean)

# 3. Colonnes finales utiles pour la suite
df_clean = df_clean[["id", "label", "path_str", "path_labels", "path_ids", "depth"]].copy()
df_clean["path_id_num"] = range(1, len(df_clean) + 1)  # simple index technique, PAS une notion d'importance

print(f"Entrées avant dédup (après filtrage vides/ids) : {n_before_dedup}")
print(f"Entrées après dédup par path_str                : {n_after_dedup}")
print(f"Chemins retirés (doublons)                       : {n_before_dedup - n_after_dedup}")
print(f"\n>>> NOMBRE FINAL DE CHEMINS VALIDES POUR LE DATASET : {len(df_clean)} <<<")

Entrées avant dédup (après filtrage vides/ids) : 1383
Entrées après dédup par path_str                : 1382
Chemins retirés (doublons)                       : 1

>>> NOMBRE FINAL DE CHEMINS VALIDES POUR LE DATASET : 1382 <<<


In [20]:
print("Aperçu du DataFrame propre (df_clean) :")
display(df_clean.sample(min(10, len(df_clean)), random_state=42)[["id", "label", "path_str", "depth"]])

Aperçu du DataFrame propre (df_clean) :


,id,label,path_str,depth
309,PDT_REF_027T_1,Gestion des Types d'Interface avec PREMIUM,Produits > Referentiel > Interface > Gestion des Types d'Interface avec PREMIUM,4
741,PDT_PAR_001T_21_154,Gestion des Types de Collectivite,Population > Referentiel > Collectivite > Gestion des Types de Collectivite,4
265,PDT_REF_016T_1,Gestion des Types de Niveau de rappel,Produits > Referentiel > General > Gestion des Types de Niveau de rappel,4
823,POP_ADR_013T_1,Gestion des Categories de Secteur,Population > Adresse > Gestion des Categories de Secteur,3
778,PDT_PAR_001T_21_427,Gestion des Types d'Expediteur de Mail,Population > Referentiel > Gestion des Mails > Gestion des Types d'Expediteur de Mail,4
660,PDT_PAR_001T_21_336,Gestion des Types de Portefeuille,Population > Referentiel > General > Gestion des Types de Portefeuille,4
76,PDT_PAR_001T_21_237,Gestion des Types de Lot de Produit GRC,G.R.C. > Referentiel Ventes > Devis > Gestion des Types de Lot de Produit GRC,4
184,INT_INT_006T_1,De-doublonnage de Collectivite,G.R.C. > Integration > De-doublonnage de Collectivite,3
745,PDT_PAR_001T_21_308,Gestion des Anomalies de Repercussion aux Salaries,Population > Referentiel > Collectivite > Gestion des Anomalies de Repercussion aux Salaries,4
486,PDT_PAR_001T_21_4,Modes de Fixation des Tarifs,Prestations > Referentiel > Professionnels de Sante > Modes de Fixation des Tarifs,4


## 1.6 — Résumé final de l'étape 1

Ce résumé doit être vérifié avant de passer à l'Étape 2 (construction du
dataset supervisé). En particulier, comparez le nombre final de chemins
valides au nombre attendu (~1383) : un écart important doit être investigué
avant de continuer.

In [21]:
summary = {
    "total_entries_raw": int(n_total),
    "schema_issues": len(schema_issues),
    "empty_paths": int(len(empty_paths)),
    "shallow_paths_depth_leq_1": int(len(shallow_paths)),
    "missing_ids": int(len(missing_ids)),
    "duplicated_ids_distinct": int(len(duplicated_ids)),
    "duplicated_paths_distinct": int(len(duplicated_paths)),
    "inconsistent_entries": int(len(inconsistent)),
    "depth_min": int(df["depth"].min()),
    "depth_max": int(df["depth"].max()),
    "final_valid_paths_for_dataset": int(len(df_clean)),
}

print("=" * 60)
print("RÉSUMÉ ÉTAPE 1")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:40s}: {v}")
print("=" * 60)

# Sauvegarde pour l'étape 2 (à ne PAS committer dans l'app existante)
df_clean.to_json("menu_index_clean.json", orient="records", force_ascii=False, indent=2)
df_clean.to_csv("menu_index_clean.csv", index=False)
print("\nFichiers sauvegardés : menu_index_clean.json, menu_index_clean.csv")
print("Ces fichiers serviront de source pour l'Étape 2 (construction du dataset).")

RÉSUMÉ ÉTAPE 1
total_entries_raw                       : 1383
schema_issues                           : 0
empty_paths                             : 0
shallow_paths_depth_leq_1               : 1
missing_ids                             : 0
duplicated_ids_distinct                 : 38
duplicated_paths_distinct               : 1
inconsistent_entries                    : 0
depth_min                               : 1
depth_max                               : 4
final_valid_paths_for_dataset           : 1382

Fichiers sauvegardés : menu_index_clean.json, menu_index_clean.csv
Ces fichiers serviront de source pour l'Étape 2 (construction du dataset).


In [22]:
import os

print(os.listdir("/content"))

['.config', 'models', 'train.csv', 'dataset_triplets.csv', 'menu_index.json', 'menu_paths_clean.csv', 'validation.csv', 'dataset_query_positive.csv', 'menu_index_clean.csv', 'test.csv', 'menu_index_clean.json', 'menu_paths_clean.json', 'sample_data']


***ETAPE 2 : Dataset supervise propre (base query->positiv)***

**Objectif :** dédupliquer les 1383→1382 chemins et construire un DataFrame stable avec un path_id numérique 1..1382 qui servira de clé pivot pour tout le reste (formulations, hard negatives, split). Ce path_id est un simple index technique, pas une mesure d'importance — c'est ce qui garantit plus tard que le chemin #1382 est traité exactement comme le #1, conformément à votre exigence critique.

In [23]:
import json, pandas as pd

with open("menu_index.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
df["depth"] = df["path_labels"].apply(len)
df["leaf_label"] = df["path_labels"].apply(lambda x: x[-1])
df["parent_labels"] = df["path_labels"].apply(lambda x: x[:-1])

# Dédoublonnage sur path_str (garde la 1ère occurrence, log le reste)
dup_mask = df.duplicated(subset="path_str", keep="first")
print("Doublons supprimés :", dup_mask.sum())
print(df[dup_mask][["id", "path_str"]])

df_clean = df[~dup_mask].reset_index(drop=True)
df_clean.insert(0, "path_id", range(1, len(df_clean) + 1))  # pivot stable, PAS un score d'importance

assert len(df_clean) == 1382, f"Attendu 1382, obtenu {len(df_clean)}"
df_clean.to_json("menu_paths_clean.json", orient="records", force_ascii=False, indent=2)
df_clean.to_csv("menu_paths_clean.csv", index=False)

print(f"✅ {len(df_clean)} chemins uniques prêts pour le dataset.")
df_clean[["path_id","id","leaf_label","path_str","depth"]].sample(5)

Doublons supprimés : 1
                  id  \
1294  PRL_MAN_054T_3   

                                                                                            path_str  
1294  Dossiers Medicaux Administratifs > Gestion des DMA > Validation Manuelle des Prises en Charges  
✅ 1382 chemins uniques prêts pour le dataset.


,path_id,id,leaf_label,path_str,depth
309,310,PDT_REF_027T_1,Gestion des Types d'Interface avec PREMIUM,Produits > Referentiel > Interface > Gestion des Types d'Interface avec PREMIUM,4
741,742,PDT_PAR_001T_21_154,Gestion des Types de Collectivite,Population > Referentiel > Collectivite > Gestion des Types de Collectivite,4
265,266,PDT_REF_016T_1,Gestion des Types de Niveau de rappel,Produits > Referentiel > General > Gestion des Types de Niveau de rappel,4
823,824,POP_ADR_013T_1,Gestion des Categories de Secteur,Population > Adresse > Gestion des Categories de Secteur,3
778,779,PDT_PAR_001T_21_427,Gestion des Types d'Expediteur de Mail,Population > Referentiel > Gestion des Mails > Gestion des Types d'Expediteur de Mail,4


***ÉTAPE 3 — Génération de formulations (couverture 100% des 1382 chemins)***

**Objectif :** générer plusieurs requêtes naturelles par chemin, pour les 1382 chemins sans exception. Vu le volume (1382 chemins × 6-8 formulations = ~9000-11000 requêtes), une génération par LLM externe serait lente/coûteuse et invérifiable à l'échelle. Je propose une génération par templates paramétrés (8 types que vous avez décrits) avec variation lexicale (synonymes de verbes) contrôlée par seed — 100% reproductible, gratuite, et surtout garantie de couvrir chaque chemin puisqu'elle est appliquée en boucle sur df_clean sans dépendre d'une API qui pourrait timeout ou sauter des lignes.

In [24]:
import random
random.seed(42)

VERBES_ACTION = ["gérer", "administrer", "paramétrer", "modifier", "traiter"]
VERBES_ACCES  = ["accéder à", "consulter", "trouver", "localiser", "voir"]

def clean(txt):
    return " ".join(txt.strip().split())

def generate_queries(row):
    leaf = row["leaf_label"]
    parent = row["parent_labels"][-1] if row["parent_labels"] else ""
    v1, v2 = random.choice(VERBES_ACTION), random.choice(VERBES_ACCES)
    v1b = random.choice([x for x in VERBES_ACTION if x != v1])

    templates = [
        f"je veux {v1} {leaf.lower()}",                                            # direct
        f"où puis-je {v2} {leaf.lower()} ?",                                       # naturelle
        f"je cherche l'écran qui permet de {v1} {leaf.lower()}" + (f" dans {parent.lower()}" if parent else ""),  # longue
        f"où trouver {leaf.lower()} ?",                                            # synonyme court
        f"j'ai besoin d'accéder à la fonctionnalité {leaf.lower()}",               # éloignée du libellé
        f"{leaf.lower()}",                                                         # très courte
        f"je dois {v1b} {leaf.lower()}, je vais où ?",                             # conversationnelle
        f"comment {v2} {leaf.lower()} ?",                                          # orientée tâche
    ]
    return list(dict.fromkeys(clean(t) for t in templates))  # dédup intra-chemin en gardant l'ordre

rows = []
for _, r in df_clean.iterrows():
    for q in generate_queries(r):
        rows.append({"path_id": r["path_id"], "id": r["id"],
                     "query": q, "positive": r["path_str"]})

dataset_df = pd.DataFrame(rows).drop_duplicates(subset=["path_id", "query"])
dataset_df.to_csv("dataset_query_positive.csv", index=False)
print(f"✅ {len(dataset_df)} exemples query→positive générés pour {dataset_df['path_id'].nunique()} chemins.")

✅ 11056 exemples query→positive générés pour 1382 chemins.


In [25]:
print("=== CHEMIN 1382 ===")

print(
    df_clean[
        df_clean["path_id"] == 1294
    ][
        ["path_id", "id", "path_str", "depth"]
    ].to_string(index=False)
)

print("\n=== QUERIES DU CHEMIN 1382 ===")

print(
    dataset_df[
        dataset_df["path_id"] == 1294
    ][
        ["path_id", "query", "positive"]
    ].to_string(index=False)
)

=== CHEMIN 1382 ===
 path_id             id                                                                                       path_str  depth
    1294 PRL_MAN_054T_1 Dossiers Medicaux Administratifs > Gestion des DMA > Validation Manuelle des Prises en Charges      3

=== QUERIES DU CHEMIN 1382 ===
 path_id                                                                                                    query                                                                                       positive
    1294                                               je veux modifier validation manuelle des prises en charges Dossiers Medicaux Administratifs > Gestion des DMA > Validation Manuelle des Prises en Charges
    1294                                         où puis-je accéder à validation manuelle des prises en charges ? Dossiers Medicaux Administratifs > Gestion des DMA > Validation Manuelle des Prises en Charges
    1294 je cherche l'écran qui permet de modifier validation manuell

# ***RAPPORT DE COUVERTURE DU DATASET***

In [26]:
# ============================================================
# RAPPORT DE COUVERTURE DU DATASET
# ============================================================

print("=" * 70)
print("📊 RAPPORT DE COUVERTURE DU DATASET")
print("=" * 70)

# ------------------------------------------------------------
# 1. Nombre total de chemins
# ------------------------------------------------------------

total_paths = len(df_clean)

# ------------------------------------------------------------
# 2. Nombre de chemins représentés dans le dataset
# ------------------------------------------------------------

covered_paths = dataset_df["path_id"].nunique()

# ------------------------------------------------------------
# 3. Identifier les chemins manquants
# ------------------------------------------------------------

all_path_ids = set(df_clean["path_id"])
covered_path_ids = set(dataset_df["path_id"])

missing_paths = sorted(
    all_path_ids - covered_path_ids
)

# ------------------------------------------------------------
# 4. Calcul de la couverture
# ------------------------------------------------------------

coverage = (
    covered_paths / total_paths * 100
)

# ------------------------------------------------------------
# 5. Affichage principal
# ------------------------------------------------------------

print(f"Nombre total de chemins      : {total_paths}")
print(f"Chemins représentés          : {covered_paths}")
print(f"Chemins manquants            : {len(missing_paths)}")
print(f"Couverture                   : {coverage:.2f}%")

# ------------------------------------------------------------
# 6. Vérification 100 %
# ------------------------------------------------------------

if len(missing_paths) == 0:
    print("\n✅ EXCELLENT : 100% DES CHEMINS SONT REPRÉSENTÉS.")
else:
    print("\n❌ ATTENTION : certains chemins sont absents.")
    print("Chemins manquants :", missing_paths)

# ------------------------------------------------------------
# 7. Nombre de queries par chemin
# ------------------------------------------------------------

query_counts = (
    dataset_df
    .groupby("path_id")
    .size()
)

print("\n" + "=" * 70)
print("📌 DISTRIBUTION DES QUERIES PAR CHEMIN")
print("=" * 70)

print(query_counts.describe())

# ------------------------------------------------------------
# 8. Chemins ayant le moins de queries
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("⚠️ CHEMINS AVEC LE MOINS DE QUERIES")
print("=" * 70)

least_queries = (
    query_counts
    .sort_values()
    .head(20)
)

print(least_queries)

# ------------------------------------------------------------
# 9. Vérifier les chemins ayant moins de 5 queries
# ------------------------------------------------------------

low_coverage = query_counts[
    query_counts < 5
]

print("\n" + "=" * 70)
print("⚠️ CHEMINS AVEC MOINS DE 5 QUERIES")
print("=" * 70)

print(f"Nombre de chemins concernés : {len(low_coverage)}")

if len(low_coverage) > 0:
    print(low_coverage)
else:
    print("✅ Aucun chemin n'a moins de 5 queries.")

# ------------------------------------------------------------
# 10. Vérifier le premier et le dernier chemin
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🔎 VÉRIFICATION DES CHEMINS EXTRÊMES")
print("=" * 70)

for path_id in [1, total_paths]:

    print(f"\n--- PATH ID : {path_id} ---")

    path_info = df_clean[
        df_clean["path_id"] == path_id
    ]

    print(
        path_info[
            ["path_id", "id", "path_str", "depth"]
        ].to_string(index=False)
    )

    print("\nQueries générées :")

    queries = dataset_df[
        dataset_df["path_id"] == path_id
    ]

    print(
        queries[
            ["query", "positive"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 11. Vérification des doublons query
# ------------------------------------------------------------

duplicate_queries = dataset_df.duplicated(
    subset=["path_id", "query"]
).sum()

print("\n" + "=" * 70)
print("🔎 CONTRÔLE DES DOUBLONS")
print("=" * 70)

print(
    f"Doublons query/path détectés : {duplicate_queries}"
)

# ------------------------------------------------------------
# 12. Résumé final
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎯 RÉSUMÉ FINAL")
print("=" * 70)

print(f"Chemins uniques              : {total_paths}")
print(f"Chemins couverts             : {covered_paths}")
print(f"Chemins manquants            : {len(missing_paths)}")
print(f"Couverture                   : {coverage:.2f}%")
print(f"Nombre total de queries      : {len(dataset_df)}")
print(
    f"Nombre moyen de queries/path : "
    f"{query_counts.mean():.2f}"
)

if coverage == 100 and len(low_coverage) == 0:
    print("\n🟢 DATASET COUVERTURE OK")
else:
    print("\n🟠 DATASET À AMÉLIORER AVANT L'ÉTAPE 4")

📊 RAPPORT DE COUVERTURE DU DATASET
Nombre total de chemins      : 1382
Chemins représentés          : 1382
Chemins manquants            : 0
Couverture                   : 100.00%

✅ EXCELLENT : 100% DES CHEMINS SONT REPRÉSENTÉS.

📌 DISTRIBUTION DES QUERIES PAR CHEMIN
count    1382.0
mean        8.0
std         0.0
min         8.0
25%         8.0
50%         8.0
75%         8.0
max         8.0
dtype: float64

⚠️ CHEMINS AVEC LE MOINS DE QUERIES
path_id
1369    8
1370    8
1371    8
1372    8
1373    8
1374    8
1375    8
1376    8
1361    8
1362    8
1363    8
1364    8
1365    8
1366    8
1367    8
1368    8
1353    8
1354    8
1355    8
1356    8
dtype: int64

⚠️ CHEMINS AVEC MOINS DE 5 QUERIES
Nombre de chemins concernés : 0
✅ Aucun chemin n'a moins de 5 queries.

🔎 VÉRIFICATION DES CHEMINS EXTRÊMES

--- PATH ID : 1 ---
 path_id             id           path_str  depth
       1 NEM_OUT_004T_1 Parametrage > Test      2

Queries générées :
                                              

In [27]:
# ============================================================
# ÉTAPE 3.5 — QUALITY CHECK DU DATASET
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter

print("=" * 70)
print("🔍 ÉTAPE 3.5 — QUALITY CHECK")
print("=" * 70)


# ============================================================
# 1. Vérification générale
# ============================================================

print("\n[1] VÉRIFICATION GÉNÉRALE")
print("-" * 70)

print(f"Nombre de chemins       : {df_clean['path_id'].nunique()}")
print(f"Nombre de queries      : {len(dataset_df)}")
print(f"Nombre de positives    : {dataset_df['positive'].nunique()}")

assert df_clean["path_id"].nunique() == 1382
assert dataset_df["path_id"].nunique() == 1382

print("✅ 1382 chemins présents dans le dataset")


# ============================================================
# 2. Vérification couverture
# ============================================================

print("\n[2] COUVERTURE")
print("-" * 70)

paths_menu = set(df_clean["path_id"])
paths_dataset = set(dataset_df["path_id"])

missing = paths_menu - paths_dataset
extra = paths_dataset - paths_menu

print(f"Chemins manquants : {len(missing)}")
print(f"Chemins inconnus  : {len(extra)}")

assert len(missing) == 0
assert len(extra) == 0

print("✅ Couverture 100 %")


# ============================================================
# 3. Nombre de queries par chemin
# ============================================================

print("\n[3] DISTRIBUTION DES QUERIES")
print("-" * 70)

queries_per_path = dataset_df.groupby("path_id").size()

print(queries_per_path.describe())

print(
    f"\nMinimum : {queries_per_path.min()}"
)
print(
    f"Maximum : {queries_per_path.max()}"
)

assert queries_per_path.min() >= 8

print("✅ Chaque chemin possède au moins 8 queries")


# ============================================================
# 4. Doublons exacts query/path
# ============================================================

print("\n[4] DOUBLONS QUERY / PATH")
print("-" * 70)

duplicates = dataset_df.duplicated(
    subset=["path_id", "query"]
).sum()

print(f"Doublons : {duplicates}")

assert duplicates == 0

print("✅ Aucun doublon query/path")


# ============================================================
# 5. Queries identiques associées à plusieurs chemins
# ============================================================

print("\n[5] QUERIES IDENTIQUES ENTRE PLUSIEURS CHEMINS")
print("-" * 70)

query_path_counts = (
    dataset_df
    .groupby("query")["path_id"]
    .nunique()
)

ambiguous_queries = query_path_counts[
    query_path_counts > 1
]

print(
    f"Queries associées à plusieurs chemins : "
    f"{len(ambiguous_queries)}"
)

if len(ambiguous_queries) > 0:

    print("\n⚠️ Exemples :")

    for q in ambiguous_queries.head(10).index:

        print(f"\nQuery : {q}")

        print(
            dataset_df[
                dataset_df["query"] == q
            ][
                ["path_id", "positive"]
            ].drop_duplicates().to_string(index=False)
        )

else:

    print("✅ Aucune query ambiguë exacte")


# ============================================================
# 6. Vérification des labels très courts
# ============================================================

print("\n[6] LABELS TRÈS COURTS")
print("-" * 70)

short_labels = df_clean[
    df_clean["leaf_label"].str.len() <= 4
][
    ["path_id", "leaf_label", "path_str"]
]

print(
    f"Nombre de chemins avec label <= 4 caractères : "
    f"{len(short_labels)}"
)

print("\nExemples :")

print(
    short_labels.head(20).to_string(index=False)
)


# ============================================================
# 7. Vérification des labels identiques sur plusieurs chemins
# ============================================================

print("\n[7] LEAF LABELS PARTAGÉS")
print("-" * 70)

leaf_counts = (
    df_clean
    .groupby("leaf_label")["path_id"]
    .nunique()
)

shared_leaves = leaf_counts[
    leaf_counts > 1
].sort_values(ascending=False)

print(
    f"Nombre de leaf labels présents sur plusieurs chemins : "
    f"{len(shared_leaves)}"
)

if len(shared_leaves) > 0:

    print("\nTop des labels partagés :")

    print(shared_leaves.head(20))


# ============================================================
# 8. Vérification de la profondeur
# ============================================================

print("\n[8] PROFONDEUR DES CHEMINS")
print("-" * 70)

print(
    df_clean["depth"].describe()
)

print(
    f"\nProfondeur minimale : {df_clean['depth'].min()}"
)

print(
    f"Profondeur maximale : {df_clean['depth'].max()}"
)


# ============================================================
# 9. Vérification des chemins très longs
# ============================================================

print("\n[9] CHEMINS PROFONDS")
print("-" * 70)

deep_paths = df_clean[
    df_clean["depth"] >= 5
][
    ["path_id", "path_str", "depth"]
].sort_values(
    "depth",
    ascending=False
)

print(
    f"Nombre de chemins avec profondeur >= 5 : "
    f"{len(deep_paths)}"
)

print("\nTop 10 des chemins les plus profonds :")

print(
    deep_paths.head(10).to_string(index=False)
)


# ============================================================
# 10. Vérification des queries réellement différentes
# ============================================================

print("\n[10] DIVERSITÉ DES QUERIES")
print("-" * 70)

unique_queries = dataset_df["query"].nunique()

print(
    f"Queries totales      : {len(dataset_df)}"
)

print(
    f"Queries uniques      : {unique_queries}"
)

print(
    f"Taux d'unicité global : "
    f"{unique_queries / len(dataset_df) * 100:.2f}%"
)


# ============================================================
# 11. Vérification du chemin 1382
# ============================================================

print("\n[11] TEST DU DERNIER CHEMIN — PATH 1382")
print("-" * 70)

last_path = df_clean[
    df_clean["path_id"] == 1382
]

print(
    last_path[
        ["path_id", "id", "leaf_label", "path_str", "depth"]
    ].to_string(index=False)
)

print("\nQueries associées :")

print(
    dataset_df[
        dataset_df["path_id"] == 1382
    ][
        ["query", "positive"]
    ].to_string(index=False)
)


# ============================================================
# 12. Résumé final
# ============================================================

print("\n" + "=" * 70)
print("🎯 RÉSUMÉ DU QUALITY CHECK")
print("=" * 70)

print("✅ 1382 chemins présents")
print("✅ Couverture 100 %")
print("✅ Minimum 8 queries par chemin")
print("✅ Aucun doublon query/path")
print("✅ Vérification des queries ambiguës effectuée")
print("✅ Vérification des labels courts effectuée")
print("✅ Vérification des labels partagés effectuée")
print("✅ Vérification de la profondeur effectuée")
print("✅ Chemin 1382 vérifié")

print("\n🟢 ÉTAPE 3.5 TERMINÉE")
print("➡️ Prêt pour l'ÉTAPE 4 — HARD NEGATIVES")
print("=" * 70)

🔍 ÉTAPE 3.5 — QUALITY CHECK

[1] VÉRIFICATION GÉNÉRALE
----------------------------------------------------------------------
Nombre de chemins       : 1382
Nombre de queries      : 11056
Nombre de positives    : 1382
✅ 1382 chemins présents dans le dataset

[2] COUVERTURE
----------------------------------------------------------------------
Chemins manquants : 0
Chemins inconnus  : 0
✅ Couverture 100 %

[3] DISTRIBUTION DES QUERIES
----------------------------------------------------------------------
count    1382.0
mean        8.0
std         0.0
min         8.0
25%         8.0
50%         8.0
75%         8.0
max         8.0
dtype: float64

Minimum : 8
Maximum : 8
✅ Chaque chemin possède au moins 8 queries

[4] DOUBLONS QUERY / PATH
----------------------------------------------------------------------
Doublons : 0
✅ Aucun doublon query/path

[5] QUERIES IDENTIQUES ENTRE PLUSIEURS CHEMINS
----------------------------------------------------------------------
Queries associées à plu

***ÉTAPE 4 — Hard negatives (étape critique)***

**Objectif :** pour chaque requête, sélectionner 1-2 chemins concurrents proches mais faux, pour que le modèle apprenne à trancher entre chemins similaires (même parent, mots partagés, libellés proches) plutôt que face à des négatifs trop faciles (totalement différents).

***Stratégie retenue (priorité décroissante) :***

Même leaf_label, parent différent — collision de libellé pure, le cas le plus piégeux.
Même parent immédiat — chemins "frères" dans la même catégorie.
Chevauchement lexical fort (indice de Jaccard sur les tokens du path_str) — chemins qui partagent du vocabulaire métier sans être frères.
Repli (fallback) aléatoire hors de la branche — uniquement si aucun candidat pertinent n'existe, et signalé dans le rapport (cas des libellés uniques comme "Sortie").

Un index inversé mot→chemins évite un calcul en O(n²) sur 1382 chemins.

In [28]:
import re
from collections import defaultdict

STOPWORDS = {"de","des","du","la","le","les","et","en","à","d","l","un","une","au","aux"}

def tokenize(path_str):
    words = re.findall(r"[a-zàâäéèêëïîôöùûüç]+", path_str.lower())
    return set(w for w in words if w not in STOPWORDS and len(w) > 2)

df_clean["tokens"] = df_clean["path_str"].apply(tokenize)
df_clean["parent_str"] = df_clean["parent_labels"].apply(lambda x: " > ".join(x))

leaf_index   = defaultdict(list)
parent_index = defaultdict(list)
word_index   = defaultdict(set)

for _, r in df_clean.iterrows():
    leaf_index[r["leaf_label"].lower()].append(r["path_id"])
    parent_index[r["parent_str"]].append(r["path_id"])
    for w in r["tokens"]:
        word_index[w].add(r["path_id"])

path_lookup = df_clean.set_index("path_id")

def get_hard_negatives(path_id, k=3):
    row = path_lookup.loc[path_id]
    candidates, seen = [], {path_id}

    for pid in leaf_index[row["leaf_label"].lower()]:
        if pid not in seen:
            candidates.append((pid, 3)); seen.add(pid)   # priorité max

    for pid in parent_index[row["parent_str"]]:
        if pid not in seen:
            candidates.append((pid, 2)); seen.add(pid)

    jaccard_pool = set()
    for w in row["tokens"]:
        jaccard_pool |= word_index[w]
    jaccard_pool -= seen
    scored = []
    for pid in jaccard_pool:
        other = path_lookup.loc[pid]["tokens"]
        j = len(row["tokens"] & other) / max(1, len(row["tokens"] | other))
        scored.append((pid, j))
    scored.sort(key=lambda x: -x[1])
    for pid, j in scored[:k]:
        candidates.append((pid, 1 + j)); seen.add(pid)

    candidates.sort(key=lambda x: -x[1])
    result = [pid for pid, _ in candidates[:k]]

    used_fallback = False
    if len(result) < k:
        pool = [p for p in df_clean["path_id"] if p not in seen | set(result)]
        extra = random.sample(pool, min(k - len(result), len(pool)))
        result += extra
        used_fallback = True

    return result, used_fallback

# Construction du dataset final (query, positive, negative)
random.seed(42)
triplets, fallback_paths = [], set()

for path_id, group in dataset_df.groupby("path_id"):
    negs, used_fb = get_hard_negatives(path_id, k=3)
    neg_paths = [path_lookup.loc[n]["path_str"] for n in negs]
    if used_fb:
        fallback_paths.add(path_id)
    for i, (_, row) in enumerate(group.iterrows()):
        triplets.append({
            "path_id": path_id,
            "query": row["query"],
            "positive": row["positive"],
            "negative": neg_paths[i % len(neg_paths)],
        })

triplet_df = pd.DataFrame(triplets)
triplet_df.to_csv("dataset_triplets.csv", index=False)
print(f"✅ {len(triplet_df)} triplets (query, positive, negative) générés.")
print(f"⚠️ Chemins ayant nécessité un negative de repli (fallback) : {len(fallback_paths)} / {df_clean.shape[0]}")

✅ 11056 triplets (query, positive, negative) générés.
⚠️ Chemins ayant nécessité un negative de repli (fallback) : 1 / 1382


In [29]:
print("df_clean existe :", "df_clean" in globals())
print("dataset_df existe :", "dataset_df" in globals())

df_clean existe : True
dataset_df existe : True


In [30]:
# 1. Aucun negative ne doit être égal au positive
bad = triplet_df[triplet_df["positive"] == triplet_df["negative"]]
assert len(bad) == 0, f"⚠️ {len(bad)} triplets avec negative == positive !"

# 2. Couverture : chaque chemin doit avoir des hard negatives
covered = triplet_df.groupby("path_id")["negative"].nunique()
print("Chemins avec au moins 1 negative distinct :", (covered >= 1).sum(), "/", df_clean.shape[0])

# 3. Exemples pour inspection manuelle (dont les cas limites "Test" / "Sortie")
print(triplet_df[triplet_df["path_id"] == 1].to_string(index=False))
print(triplet_df[triplet_df["path_id"] == 1382].to_string(index=False))

print("🟢 HARD NEGATIVES OK" if len(bad) == 0 else "🔴 PROBLÈME DÉTECTÉ")

Chemins avec au moins 1 negative distinct : 1382 / 1382
 path_id                                                        query           positive                             negative
       1                                           je veux gérer test Parametrage > Test      Parametrage > Regles de gestion
       1                                  où puis-je accéder à test ? Parametrage > Test                  Parametrage > Alias
       1 je cherche l'écran qui permet de gérer test dans parametrage Parametrage > Test Parametrage > Parametres applicatifs
       1                                            où trouver test ? Parametrage > Test      Parametrage > Regles de gestion
       1               j'ai besoin d'accéder à la fonctionnalité test Parametrage > Test                  Parametrage > Alias
       1                                                         test Parametrage > Test Parametrage > Parametres applicatifs
       1                          je dois modifier test, je va

***ÉTAPE 5 — Split train / validation / test***

**Objectif :** obtenir 80/10/10 sans fuite de données, tout en respectant votre exigence critique : chaque chemin (y compris #1382) doit avoir au moins une query de test.

Le vrai risque de leakage ici n'est pas "même chemin dans train et test" — c'est attendu et voulu (le modèle doit reconnaître le chemin #1382 qu'il ait vu 6 ou 8 formulations dessus). Le vrai risque, c'est la fuite au niveau des formulations trop proches : si "je veux gérer test" est en train et "je veux gérer test" (quasi identique) en test, l'évaluation est artificiellement facile.

**Stratégie retenue :** split par type de template, pas aléatoire. Chaque chemin a 8 formulations de styles différents (direct, naturelle, longue, synonyme, éloignée, courte, conversationnelle, orientée tâche). Je fixe :

train = 6 types les plus littéraux (direct, naturelle, longue, synonyme_court, courte, conversationnelle) → 75%
validation = type "éloignée" (le plus différent du libellé) → 12,5%
test = type "orientée tâche" → 12,5%

Ainsi le test évalue le modèle sur un style de phrasé qu'il n'a jamais vu pour aucun chemin, avec 1 query de test garantie par chemin → couverture test 100% automatique.

In [31]:
TEMPLATE_NAMES = ["direct","naturelle","longue","synonyme_court","eloignee","courte","conversationnelle","orientee_tache"]

# Retrouve le type de template à partir de l'ordre de génération (préservé, confirmé par std=0 sur 8/chemin)
dataset_df["template_idx"]  = dataset_df.groupby("path_id").cumcount()
dataset_df["template_type"] = dataset_df["template_idx"].map(lambda i: TEMPLATE_NAMES[i])

TRAIN_TYPES = {"direct","naturelle","longue","synonyme_court","courte","conversationnelle"}
VAL_TYPES   = {"eloignee"}
TEST_TYPES  = {"orientee_tache"}

def assign_split(t):
    if t in TRAIN_TYPES: return "train"
    if t in VAL_TYPES:   return "validation"
    return "test"

dataset_df["split"] = dataset_df["template_type"].apply(assign_split)

# Rattache le split aux triplets (jointure sur path_id + query, identiques entre les deux DataFrames)
triplet_df = triplet_df.merge(
    dataset_df[["path_id", "query", "split", "template_type"]],
    on=["path_id", "query"], how="left"
)
assert triplet_df["split"].isna().sum() == 0, "⚠️ Des triplets n'ont pas pu être rattachés à un split"

train_df = triplet_df[triplet_df.split == "train"].drop(columns=["split"])
val_df   = triplet_df[triplet_df.split == "validation"].drop(columns=["split"])
test_df  = triplet_df[triplet_df.split == "test"].drop(columns=["split"])

train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")

Train: 8292 | Validation: 1382 | Test: 1382


In [32]:
# 1. Aucune query identique entre splits (pas de fuite au niveau texte)
assert set(train_df["query"]) & set(test_df["query"])  == set()
assert set(train_df["query"]) & set(val_df["query"])   == set()
assert set(val_df["query"])   & set(test_df["query"])  == set()
print("✅ Aucune fuite de query entre train / validation / test.")

# 2. Couverture totale : chaque chemin a EXACTEMENT 1 query test et 1 query validation
assert test_df["path_id"].nunique() == 1382, f"Manque {1382 - test_df['path_id'].nunique()} chemins en test"
assert val_df["path_id"].nunique()  == 1382, f"Manque {1382 - val_df['path_id'].nunique()} chemins en validation"
print("✅ Les 1382 chemins ont au moins 1 query de test et 1 query de validation (y compris path_id 1382).")

# 3. Proportions réelles
n = len(triplet_df)
print(f"Proportions : train {len(train_df)/n*100:.1f}% / val {len(val_df)/n*100:.1f}% / test {len(test_df)/n*100:.1f}%")

# 4. Vérification explicite du chemin le plus "profond dans le fichier"
print(test_df[test_df.path_id == 1382][["path_id","query","positive","negative","template_type"]].to_string(index=False))

✅ Aucune fuite de query entre train / validation / test.
✅ Les 1382 chemins ont au moins 1 query de test et 1 query de validation (y compris path_id 1382).
Proportions : train 75.0% / val 12.5% / test 12.5%
 path_id                 query positive                                               negative  template_type
    1382 comment voir sortie ?   Sortie Decisionnel > Gestion des Formats de Sortie de Rapport orientee_tache


***ÉTAPE 6 — Fine-tuning du Sentence Transformer***

**Objectif :** spécialiser paraphrase-multilingual-MiniLM-L12-v2 sur vos 1382 chemins tout en gardant un modèle compatible model.encode(...) pour ChromaDB.

**Strategie:**

***Loss =*** MultipleNegativesRankingLoss (MNRL), alimentée par vos triplets (query, positive, negative). C'est la loss de référence pour le retrieval/recherche sémantique (c'est celle utilisée pour entraîner la plupart des modèles Sentence-Transformers de recherche). Contrairement à une classification, elle n'apprend pas des catégories fixes : elle rapproche l'embedding de la query de son positive et l'éloigne du negative et de tous les autres positives du batch (négatifs "gratuits" en plus de vos hard negatives explicites) → sortie finale = toujours un vecteur, donc 100% compatible model.encode().
***Évaluateur =*** InformationRetrievalEvaluator sur le corpus complet des 1382 chemins (pas juste une accuracy sur triplets) : à chaque évaluation, le modèle doit retrouver le bon chemin parmi les 1382, exactement le scénario réel avec ChromaDB. C'est ce qui permet de mesurer Top-1/Top-3/MRR par chemin en Étape 7.
***Hyperparamètres adaptés à Colab (GPU T4 gratuit) :*** batch_size=32 (MNRL profite d'un batch pas trop petit pour ses négatifs in-batch, mais 32 tient en mémoire T4 avec fp16), lr=2e-5 (standard pour fine-tuning de sentence-transformers, évite de "casser" les connaissances multilingues déjà apprises), epochs=6 avec load_best_model_at_end sur le MRR de validation (= early stopping implicite, pas d'overfitting inutile), warmup_ratio=0.1, seed=42 fixé partout pour reproductibilité.


#  Installation

In [33]:
!pip install -q -U "pyarrow==17.0.0" "datasets==3.2.0" "sentence-transformers[train]" accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


# Chargement des données

In [34]:
import pandas as pd, random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("validation.csv")
test_df  = pd.read_csv("test.csv")
corpus_df = pd.read_json("menu_paths_clean.json")  # les 1382 chemins, source de vérité pour le corpus

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Corpus: {len(corpus_df)}")
assert corpus_df["path_id"].nunique() == 1382

Train: 8292 | Val: 1382 | Test: 1382 | Corpus: 1382


# Dataset d'entraînement (format HuggingFace datasets)

In [35]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "anchor":   train_df["query"].tolist(),
    "positive": train_df["positive"].tolist(),
    "negative": train_df["negative"].tolist(),
})
print(train_dataset[0])

{'anchor': 'je veux gérer test', 'positive': 'Parametrage > Test', 'negative': 'Parametrage > Regles de gestion'}


# Modèle de base + Loss

In [36]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss

BASE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(BASE_MODEL)
loss  = MultipleNegativesRankingLoss(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Évaluateur : retrieval sur le corpus COMPLET des 1382 chemins

In [37]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# corpus = TOUS les chemins (1382), pas seulement ceux présents en validation
corpus = {str(pid): path for pid, path in zip(corpus_df["path_id"], corpus_df["path_str"])}

# queries de validation, indexées, avec leur bon chemin comme unique "relevant doc"
val_queries = {str(i): q for i, q in enumerate(val_df["query"])}
val_pathid_by_query = {str(i): str(pid) for i, pid in enumerate(val_df["path_id"])}
relevant_docs = {qid: {val_pathid_by_query[qid]} for qid in val_queries}

ir_evaluator = InformationRetrievalEvaluator(
    queries=val_queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="menu-validation-1382paths",
    accuracy_at_k=[1, 3, 5],
    mrr_at_k=[10],
    show_progress_bar=False,
)

# Entraînement

In [39]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

OUTPUT_DIR = "/content/models/cegedim-menu-embedding"

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=6,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_menu-validation-1382paths_cosine_mrr@10",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=ir_evaluator,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Menu-validation-1382paths Cosine Accuracy@1,Menu-validation-1382paths Cosine Accuracy@3,Menu-validation-1382paths Cosine Accuracy@5,Menu-validation-1382paths Cosine Precision@1,Menu-validation-1382paths Cosine Precision@3,Menu-validation-1382paths Cosine Precision@5,Menu-validation-1382paths Cosine Precision@10,Menu-validation-1382paths Cosine Recall@1,Menu-validation-1382paths Cosine Recall@3,Menu-validation-1382paths Cosine Recall@5,Menu-validation-1382paths Cosine Recall@10,Menu-validation-1382paths Cosine Ndcg@10,Menu-validation-1382paths Cosine Mrr@10,Menu-validation-1382paths Cosine Map@100
1,0.090057,No log,0.938495,0.998553,1.000000,0.938495,0.332851,0.200000,0.100000,0.938495,0.998553,1.000000,1.000000,0.976063,0.967680,0.967680
2,0.063502,No log,0.953690,0.999276,1.000000,0.953690,0.333092,0.200000,0.100000,0.953690,0.999276,1.000000,1.000000,0.982479,0.976302,0.976302
3,0.067898,No log,0.955137,0.999276,1.000000,0.955137,0.333092,0.200000,0.100000,0.955137,0.999276,1.000000,1.000000,0.983013,0.977026,0.977026
4,0.071080,No log,0.960926,0.999276,1.000000,0.960926,0.333092,0.200000,0.100000,0.960926,0.999276,1.000000,1.000000,0.985150,0.979920,0.979920
5,0.053491,No log,0.960926,0.999276,1.000000,0.960926,0.333092,0.200000,0.100000,0.960926,0.999276,1.000000,1.000000,0.985055,0.979800,0.979800
6,0.053302,No log,0.960926,0.999276,1.000000,0.960926,0.333092,0.200000,0.100000,0.960926,0.999276,1.000000,1.000000,0.985055,0.979800,0.979800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TrainOutput(global_step=1560, training_loss=0.09556741825281045, metrics={'train_runtime': 576.9328, 'train_samples_per_second': 86.235, 'train_steps_per_second': 2.704, 'total_flos': 0.0, 'train_loss': 0.09556741825281045, 'epoch': 6.0})

# Sauvegarde du meilleur modèle

In [40]:
model.save_pretrained(OUTPUT_DIR)
print(f"✅ Modèle sauvegardé dans {OUTPUT_DIR}")

# Contrôle qualité : le modèle rechargé doit fonctionner comme prévu par l'application
reloaded = SentenceTransformer(OUTPUT_DIR)
emb = reloaded.encode("je veux gérer les actes RO")
print("Shape embedding :", emb.shape)
assert emb.shape[0] == 384  # dimension MiniLM-L12
print("🟢 Modèle compatible model.encode() — prêt pour ChromaDB.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle sauvegardé dans /content/models/cegedim-menu-embedding


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Shape embedding : (384,)
🟢 Modèle compatible model.encode() — prêt pour ChromaDB.


***ÉTAPE 7 — Évaluation complète sur le test set (1382 chemins)***

**Objectif :** ne pas se contenter d'une métrique globale. Vérifier explicitement que chaque chemin, y compris les profonds/rares/en fin de fichier, est bien retrouvé — et identifier précisément ceux qui ne le sont pas.

# Chargement du modèle final + encodage du corpus complet

In [41]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import torch, pandas as pd, numpy as np

model = SentenceTransformer(OUTPUT_DIR)  # le modèle sauvegardé, rechargé "à froid" comme le fera l'app

corpus_df = pd.read_json("menu_paths_clean.json")
assert corpus_df["path_id"].nunique() == 1382

corpus_paths = corpus_df["path_str"].tolist()
corpus_ids   = corpus_df["path_id"].tolist()
corpus_emb   = model.encode(corpus_paths, convert_to_tensor=True, show_progress_bar=True)

test_df = pd.read_csv("test.csv")
assert test_df["path_id"].nunique() == 1382, "⚠️ Le test set ne couvre plus 1382 chemins !"
query_emb = model.encode(test_df["query"].tolist(), convert_to_tensor=True, show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

# Top-1 / Top-3 / MRR + résultat détaillé par requête

In [44]:
sims = cos_sim(query_emb, corpus_emb)                 # (n_test, 1382)
ranked = torch.argsort(sims, dim=1, descending=True).cpu()   # <-- fix : ramené sur CPU une fois pour toutes
corpus_ids_t = torch.tensor(corpus_ids)                       # déjà CPU par défaut

results = []
for i, row in test_df.reset_index(drop=True).iterrows():
    true_pid = row["path_id"]
    true_rank_pos = (corpus_ids_t[ranked[i]] == true_pid).nonzero()[0].item()
    top1_pid = corpus_ids[ranked[i][0].item()]
    top3_pids = [corpus_ids[j.item()] for j in ranked[i][:3]]
    results.append({
        "path_id": true_pid,
        "query": row["query"],
        "true_path": row["positive"],
        "predicted_top1": corpus_df.set_index("path_id").loc[top1_pid, "path_str"],
        "correct_top1": top1_pid == true_pid,
        "correct_top3": true_pid in top3_pids,
        "rank": true_rank_pos + 1,
        "reciprocal_rank": 1 / (true_rank_pos + 1),
    })

eval_df = pd.DataFrame(results)
eval_df = eval_df.merge(corpus_df[["path_id", "depth"]], on="path_id")

top1 = eval_df["correct_top1"].mean()
top3 = eval_df["correct_top3"].mean()
mrr  = eval_df["reciprocal_rank"].mean()

print("="*60)
print(f"Top-1 Accuracy : {top1*100:.2f}%")
print(f"Top-3 Accuracy : {top3*100:.2f}%")
print(f"MRR            : {mrr:.4f}")
print(f"Requêtes de test couvrant des chemins uniques : {eval_df['path_id'].nunique()} / 1382")
print("="*60)

Top-1 Accuracy : 96.16%
Top-3 Accuracy : 99.93%
MRR            : 0.9803
Requêtes de test couvrant des chemins uniques : 1382 / 1382


# Performance par profondeur (exigence : les chemins profonds ne doivent pas être pénalisés)

In [45]:
depth_report = eval_df.groupby("depth").agg(
    n=("path_id", "count"),
    top1_acc=("correct_top1", "mean"),
    top3_acc=("correct_top3", "mean"),
    mrr=("reciprocal_rank", "mean"),
).round(4)
print(depth_report)

         n  top1_acc  top3_acc     mrr
depth                                 
1        1    1.0000    1.0000  1.0000
2       96    1.0000    1.0000  1.0000
3      461    0.9675    1.0000  0.9837
4      824    0.9539    0.9988  0.9760


# Chemins jamais retrouvés (top1 faux ET absents du top-3)

In [46]:
never_found = eval_df[~eval_df["correct_top3"]]
print(f"Chemins jamais retrouvés dans le top-3 : {len(never_found)} / {len(eval_df)} ({len(never_found)/len(eval_df)*100:.2f}%)")
print(never_found[["path_id","query","true_path","predicted_top1","depth"]].to_string(index=False))

never_found.to_csv("test_never_found.csv", index=False)

Chemins jamais retrouvés dans le top-3 : 1 / 1382 (0.07%)
 path_id                                     query                                                     true_path                                    predicted_top1  depth
     152 comment localiser gestion des variables ? G.R.C. > Referentiel Marketing > Cout > Gestion des Variables Bureautique > Referentiel > Gestion des Variables      4


# Test spécifique : chemins difficiles (profonds, fin de fichier, hard negatives partagés)

In [47]:
triplet_test = pd.read_csv("test.csv")  # contient déjà "negative" = hard negative associé

hard_subset_ids = set(
    corpus_df[corpus_df["depth"] >= corpus_df["depth"].max()]["path_id"]   # profondeur max
) | set(corpus_df.sort_values("path_id").tail(200)["path_id"])              # 200 derniers du fichier

hard_eval = eval_df[eval_df["path_id"].isin(hard_subset_ids)]
print(f"Sous-ensemble difficile : {len(hard_eval)} chemins")
print(f"Top-1 sur chemins difficiles : {hard_eval['correct_top1'].mean()*100:.2f}%")
print(f"Top-3 sur chemins difficiles : {hard_eval['correct_top3'].mean()*100:.2f}%")

# Vérification explicite demandée : path_id 1382 traité comme path_id 1
for pid in [1, 1382]:
    r = eval_df[eval_df.path_id == pid].iloc[0]
    print(f"path_id {pid} -> correct_top1={r.correct_top1}, rank={r.rank}")

Sous-ensemble difficile : 993 chemins
Top-1 sur chemins difficiles : 95.47%
Top-3 sur chemins difficiles : 99.90%
path_id 1 -> correct_top1=True, rank=<bound method NDFrame.rank of path_id                                   1
query              comment accéder à test ?
true_path                Parametrage > Test
predicted_top1           Parametrage > Test
correct_top1                           True
correct_top3                           True
rank                                      1
reciprocal_rank                         1.0
depth                                     2
Name: 0, dtype: object>
path_id 1382 -> correct_top1=True, rank=<bound method NDFrame.rank of path_id                             1382
query              comment voir sortie ?
true_path                         Sortie
predicted_top1                    Sortie
correct_top1                        True
correct_top3                        True
rank                                   1
reciprocal_rank                      1.0
d

# Confusion avec les hard negatives (le modèle apprend-il vraiment à distinguer les chemins proches ?)

In [48]:
merged = eval_df.merge(triplet_test[["path_id","negative"]], on="path_id", how="left")
confused_with_hardneg = merged[
    (~merged["correct_top1"]) & (merged["predicted_top1"] == merged["negative"])
]
print(f"Erreurs où le top-1 prédit est exactement le hard negative associé : {len(confused_with_hardneg)}")
print(confused_with_hardneg[["path_id","query","true_path","predicted_top1"]].to_string(index=False))

Erreurs où le top-1 prédit est exactement le hard negative associé : 6
 path_id                                     query                                                       true_path                                                    predicted_top1
      59   comment trouver gestion des fonctions ?    G.R.C. > Referentiel Ventes > Acteur > Gestion des Fonctions          Echanges > Referentiel > Package > Gestion des Fonctions
     110  comment localiser gestion des messages ?  G.R.C. > Referentiel Services > Contact > Gestion des Messages Prestations > Referentiel > Remboursements > Gestion des Messages
     269    comment trouver gestion des messages ?        Produits > Referentiel  > General > Gestion des Messages Prestations > Referentiel > Remboursements > Gestion des Messages
     718   comment trouver gestion des fonctions ?     Population > Referentiel > Personne > Gestion des Fonctions          Echanges > Referentiel > Package > Gestion des Fonctions
    1198 comment consult

# sauvegarde

In [49]:
import os, json, shutil
from datetime import datetime

BASE_DIR = "/content/deliverable"
DATASET_DIR = f"{BASE_DIR}/dataset"
MODEL_SRC = OUTPUT_DIR  # "/content/models/cegedim-menu-embedding"
MODEL_DST = f"{BASE_DIR}/models/cegedim-menu-embedding"
REPORT_DIR = f"{BASE_DIR}/reports"

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)

# 1. Dataset — tous les fichiers intermédiaires + finaux, en un seul endroit
corpus_df.to_json(f"{DATASET_DIR}/menu_paths_clean.json", orient="records", force_ascii=False, indent=2)
corpus_df.to_csv(f"{DATASET_DIR}/menu_paths_clean.csv", index=False)
dataset_df.to_csv(f"{DATASET_DIR}/dataset_query_positive.csv", index=False)
triplet_df.to_csv(f"{DATASET_DIR}/dataset_triplets_full.csv", index=False)
train_df.to_csv(f"{DATASET_DIR}/train.csv", index=False)
val_df.to_csv(f"{DATASET_DIR}/validation.csv", index=False)
test_df.to_csv(f"{DATASET_DIR}/test.csv", index=False)

# 2. Modèle — copie propre depuis le dossier d'entraînement
if os.path.exists(MODEL_DST):
    shutil.rmtree(MODEL_DST)
shutil.copytree(MODEL_SRC, MODEL_DST)

# 3. Rapports d'évaluation générés en Étape 7
eval_df.to_csv(f"{REPORT_DIR}/test_predictions_detail.csv", index=False)
never_found.to_csv(f"{REPORT_DIR}/test_never_found.csv", index=False)
depth_report.to_csv(f"{REPORT_DIR}/depth_report.csv")

# 4. Résumé JSON — le résumé chiffré exigé par le cahier des charges
summary = {
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "modele_base": BASE_MODEL,
    "modele_final": "cegedim-menu-embedding",
    "total_chemins_bruts": 1383,
    "total_chemins_valides": 1382,
    "total_exemples_query_positive": len(dataset_df),
    "total_triplets_hard_negative": len(triplet_df),
    "chemins_en_fallback_negative": 1,
    "train_size": len(train_df),
    "validation_size": len(val_df),
    "test_size": len(test_df),
    "loss": "MultipleNegativesRankingLoss",
    "learning_rate": 2e-5,
    "batch_size": 32,
    "epochs": 6,
    "warmup_ratio": 0.1,
    "seed": 42,
    "top1_accuracy_test": round(float(top1), 4),
    "top3_accuracy_test": round(float(top3), 4),
    "mrr_test": round(float(mrr), 4),
    "chemins_jamais_retrouves_top3": int(len(never_found)),
}
with open(f"{REPORT_DIR}/resume_jour1.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))

# 5. Vérification que tout est bien sur disque avant de zipper
for path in [f"{DATASET_DIR}/train.csv", f"{DATASET_DIR}/test.csv",
             f"{MODEL_DST}/config.json", f"{REPORT_DIR}/resume_jour1.json"]:
    assert os.path.exists(path), f"⚠️ Fichier manquant : {path}"
print("✅ Tous les fichiers attendus sont présents sur disque.")

{
  "date": "2026-08-26 12:45",
  "modele_base": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "modele_final": "cegedim-menu-embedding",
  "total_chemins_bruts": 1383,
  "total_chemins_valides": 1382,
  "total_exemples_query_positive": 11056,
  "total_triplets_hard_negative": 11056,
  "chemins_en_fallback_negative": 1,
  "train_size": 8292,
  "validation_size": 1382,
  "test_size": 1382,
  "loss": "MultipleNegativesRankingLoss",
  "learning_rate": 2e-05,
  "batch_size": 32,
  "epochs": 6,
  "warmup_ratio": 0.1,
  "seed": 42,
  "top1_accuracy_test": 0.9616,
  "top3_accuracy_test": 0.9993,
  "mrr_test": 0.9803,
  "chemins_jamais_retrouves_top3": 1
}
✅ Tous les fichiers attendus sont présents sur disque.


In [50]:
shutil.make_archive("/content/cegedim-menu-embedding-jour1", "zip", BASE_DIR)

from google.colab import files
files.download("/content/cegedim-menu-embedding-jour1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
import os

model_path = "/content/models/cegedim-menu-embedding"

print(os.path.exists(model_path))
print(os.listdir(model_path))

True
['sentence_bert_config.json', 'model.safetensors', 'config_sentence_transformers.json', 'README.md', '1_Pooling', 'checkpoint-1040', 'modules.json', 'tokenizer_config.json', 'config.json', 'eval', 'checkpoint-1560', 'tokenizer.json']


In [52]:
from google.colab import drive
drive.mount('/content/drive')
shutil.copytree(BASE_DIR, "/content/drive/MyDrive/cegedim-menu-embedding-jour1", dirs_exist_ok=True)
print("✅ Copié dans Google Drive : MyDrive/cegedim-menu-embedding-jour1")

Mounted at /content/drive
✅ Copié dans Google Drive : MyDrive/cegedim-menu-embedding-jour1
